# STKI RAG Demo
Demo Retrieval Augmented Generation (RAG) untuk mata kuliah STKI.
Project ini membandingkan 3 pendekatan retrieval semantik:
1. Prompt-based (in-context LLM reasoning)
2. Vector search (embedding + cosine similarity)
3. KL-Divergence (softmax probability distribution)
Menggunakan model Google Gemini untuk embedding dan generasi jawaban.

## Cell 1: Setup
Import libraries and initialize Gemini client

In [3]:
import os
import time
import hashlib
from typing import Iterable

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from dotenv import load_dotenv
from google import genai


# Load .env and get API key
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise EnvironmentError(
        "GEMINI_API_KEY tidak ditemukan. "
        "Isi file .env dengan GEMINI_API_KEY=key-mu."
    )

client = genai.Client(api_key=GEMINI_API_KEY)

## Cell 2: Helper Functions
Dua fungsi utama yang dipakai berulang kali di notebook ini:
- `get_embedding` — ubah teks menjadi vektor (dengan retry + fallback lokal)
- `cosine_sim` — hitung cosine similarity antar dua vektor

In [4]:
def get_embedding(text: str, retries: int = 4, base_delay: float = 1.0) -> list[float]:
    """Generate embedding menggunakan Gemini dengan retry."""

    for attempt in range(retries):
        try:
            resp = client.models.embed_content(
                model="gemini-embedding-001",
                contents=text,
            )
            return resp.embeddings[0].values

        except Exception as exc:
            if attempt < retries - 1:
                time.sleep(base_delay * (2 ** attempt))
            else:
                raise RuntimeError(
                    f"Gagal membuat embedding setelah {retries} percobaan."
                ) from exc


def cosine_sim(a, b) -> float:
    """Cosine similarity between two vectors.
    
    Returns 0.0 if either vector is zero-length to avoid division by zero.
    """
    a, b = np.array(a), np.array(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

## Cell 3: Load Query and Documents
Tentukan query (pertanyaan) dan muat dokumen dari folder `dokumen/`.

In [5]:
query = "What time do I leave for school?"

def load_documents() -> dict[str, str]:
    """Read all markdown files from dokumen/ directory."""
    docs = {}
    for fname in ["D1.md", "D2.md", "D3.md"]:
        with open(os.path.join("dokumen", fname), encoding="utf-8") as f:
            docs[fname[:-3]] = f.read().strip()
    return docs

DOCS = load_documents()

print("Query:", query)
print("Documents loaded:", list(DOCS.keys()))
for name, text in DOCS.items():
    print(f"--- {name} ---")
    print(text)
    print()

Query: What time do I leave for school?
Documents loaded: ['D1', 'D2', 'D3']
--- D1 ---
I wake up early every morning and prepare for work.  start at **7 AM** and I always leave home at around **6:30 PM** .

--- D2 ---
SMA 1 is located on **Jl. Merdeka No. 125**, in the center of the city. The campus spans over 5 hectares with modern facilities.

- **Science Track (MIPA):** Mathematics, Physics, Chemistry, Biology
- **Social Track (IPS):** Economics, Geography, Sociology, History

--- D3 ---
**Obesity** is a health condition in which a person has excessive body fat that may increase the risk of serious diseases, such as diabetes, heart disease, and high blood pressure. It is often caused by unhealthy eating habits, lack of physical activity, genetics, and lifestyle factors. Maintaining a balanced diet and exercising regularly can help prevent obesity.



In [6]:
print(DOCS)

{'D1': 'I wake up early every morning and prepare for work.  start at **7 AM** and I always leave home at around **6:30 PM** .', 'D2': 'SMA 1 is located on **Jl. Merdeka No. 125**, in the center of the city. The campus spans over 5 hectares with modern facilities.\n\n- **Science Track (MIPA):** Mathematics, Physics, Chemistry, Biology\n- **Social Track (IPS):** Economics, Geography, Sociology, History', 'D3': '**Obesity** is a health condition in which a person has excessive body fat that may increase the risk of serious diseases, such as diabetes, heart disease, and high blood pressure. It is often caused by unhealthy eating habits, lack of physical activity, genetics, and lifestyle factors. Maintaining a balanced diet and exercising regularly can help prevent obesity.'}


## Cell 4: Vector Search
Langkah-langkah:
1. Embed semua dokumen menjadi vektor.
2. Embed query menjadi vektor.
3. Hitung cosine similarity antara query dan setiap dokumen.
4. Pilih dokumen dengan skor tertinggi.

In [7]:
doc_vecs = {}
for k, v in DOCS.items():
    try:
        doc_vecs[k] = get_embedding(v)
        print(f"Embedded {k}: {len(doc_vecs[k])} dimensions")
        table = pd.DataFrame({
            "Document" : k,
            "Embedding Value" : [doc_vecs[k]]
        })
        display(table)
        
    except Exception as exc:
        print(f"Skip dokumen {k} karena error: {exc}")

if not doc_vecs:
    raise RuntimeError("Tidak ada embedding dokumen yang berhasil dibuat.")

q_vec = get_embedding(query)
print(f"Query embedded: {len(q_vec)} dimensions")
print()

scores = {k: cosine_sim(q_vec, v) for k, v in doc_vecs.items()}
best = max(scores, key=scores.get)

print("Similarity scores:")
for k, v in sorted(scores.items(), key=lambda x: x[1], reverse=True):
    print(f"  {k}: {v:.4f}")
print()
print(f"Best match: {best}")


Embedded D1: 3072 dimensions


,Document,Embedding Value
0,D1,"[-0.025052272, 0.0063632717, 0.024698911, -0.0..."


Embedded D2: 3072 dimensions


,Document,Embedding Value
0,D2,"[0.029824194, 0.003279845, 0.008537714, -0.041..."


Embedded D3: 3072 dimensions


,Document,Embedding Value
0,D3,"[0.023814542, -0.014168126, 0.01730474, -0.035..."


Query embedded: 3072 dimensions

Similarity scores:
  D1: 0.6864
  D2: 0.5242
  D3: 0.4645

Best match: D1


## Cell 5: RAG Answer Generation
Dengan dokumen terbaik dari cosine similarity, kirim ke Gemini untuk generate
jawaban terstruktur: ANSWER / SNIPPET / REASON.

In [8]:
METRIC = "cosine similarity"  # bisa diganti ke "KL-Divergence"

doc_text = DOCS[best]
scores_text = "".join([f"- {k}: {v:.4f}" for k, v in scores.items()])

prompt = (
    "You are a semantic Question Answering system."
    f'A user asked: "{query}"'
    f"Based on {METRIC}, the best matching document is {best}."
    f"Scores:{scores_text}"
    f"Content of {best}:{doc_text}"
    "Answer the query using ONLY the content above."
    "Format your answer as:"
    f"ANSWER: [direct answer based on {best}]"
    "SNIPPET: [1 relevant sentence from the document]"
    "REASON: [why this document is the best match in 1 sentence]"
)

print("Generating answer from", best, "...")
start_answer = time.time()
resp = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt,
)
elapsed_answer = time.time() - start_answer

# Parse structured output
lines = resp.text.strip().splitlines()
rag_result = {"answer": "", "snippet": "", "reason": ""}
current = ""
for line in lines:
    if line.startswith("ANSWER:"):
        current = "answer"
        rag_result["answer"] = line.replace("ANSWER:", "").strip()
    elif line.startswith("SNIPPET:"):
        current = "snippet"
        rag_result["snippet"] = line.replace("SNIPPET:", "").strip()
    elif line.startswith("REASON:"):
        current = "reason"
        rag_result["reason"] = line.replace("REASON:", "").strip()
    elif current == "answer" and line.strip():
        rag_result["answer"] += " " + line.strip()
    elif current == "reason" and line.strip():
        rag_result["reason"] += " " + line.strip()

print("ANSWER:", rag_result["answer"])
print("SNIPPET:", rag_result["snippet"])
print("REASON:", rag_result["reason"])
print(f"[Generated in {elapsed_answer:.2f}s]")

Generating answer from D1 ...
ANSWER: You leave for school at around 6:30 PM.
SNIPPET: I wake up early every morning and prepare for work. start at 7 AM and I always leave home at around 6:30 PM .
REASON: D1 is the best match because it contains the specific time the user leaves home, which directly answers the query about their departure time.
[Generated in 1.31s]
